# 00 · Setup — a human operator governs the catalog

This example has **two tiers of identity**, and that split is the whole point:

* **peter** — the *human* platform operator. He logs in **interactively** with the
  OAuth2 **device-authorization grant** (RFC 8628): a URL + code you approve in your
  browser. No client secret sits in the notebook. Peter is the admin — he bootstraps
  Lakekeeper, builds the warehouse, and grants everyone else's access.
* **three service accounts** — autonomous jobs with no human in the loop, so they use
  the **client-credentials** grant:
  `PIPELINE` (writes the medallion), `ANALYST` and `CONTRACTOR` (two AI agents).

> The kernel runs **inside** the docker network, so it reaches Keycloak and SeaweedFS
> directly. The device-login URL it prints is host-facing (`localhost:30080`) — open it
> in the browser on your machine.

In [ ]:
import sys, os
sys.path.insert(0, '/work')  # mlib.py, icehelp.py, clip_util.py live at the mount root
import requests
import mlib
from mlib import (CATALOG_URL, MANAGEMENT_URL, WAREHOUSE_NAME, S3_ENDPOINT,
                  NS_RAW, NS_BRONZE, NS_SILVER, NS_GOLD, PIPELINE, ANALYST, CONTRACTOR, auth)

## 1 · Log in as peter (interactive device code)

**Device-code flow** (RFC 8628) is how a *human* logs in from somewhere that can't pop open a login page — a container, a remote kernel. The notebook shows a URL + code; you approve it in your own browser and the notebook polls until you do. No password or client secret ever lives in the notebook.

Run the next cell, open the printed URL, sign in as **peter** / `iceberg`, and approve — it returns peter's token (which refreshes itself afterwards).

In [ ]:
peter = mlib.device_login()
print('subject (Lakekeeper user will be oidc~<sub>):', mlib.subject(peter.token))

## 2 · Bootstrap Lakekeeper

The first principal to call `/bootstrap` becomes the instance admin. Because peter does
it, **peter** is the admin — no static admin secret anywhere.

In [ ]:
r = requests.post(f'{MANAGEMENT_URL}/v1/bootstrap', headers=auth(peter.token),
                  json={'accept-terms-of-use': True}, timeout=15)
print('bootstrap:', r.status_code, '(409 = already bootstrapped)')

## 3 · Create the warehouse (SeaweedFS + STS)

A **warehouse** is a governed storage area. We back it with SeaweedFS (a local, S3-compatible store) and turn on **STS credential vending** (`sts-enabled: true`) — the mechanism the whole demo hinges on. Instead of ever handing a client the long-lived root key, Lakekeeper mints **short-lived, prefix-scoped credentials on demand**, and only for principals it has authorized. Every read/write below — and every agent later — goes through those vended credentials.

In [ ]:
warehouse = {
    'warehouse-name': WAREHOUSE_NAME,
    'storage-profile': {
        'type': 's3', 'bucket': 'medallion', 'key-prefix': 'warehouse',
        'endpoint': S3_ENDPOINT, 'sts-endpoint': S3_ENDPOINT,  # LAN IP when started via ./up.sh
        'sts-role-arn': 'arn:aws:iam::000000000000:role/LakekeeperVendedRole',
        'region': 'local-01', 'path-style-access': True,
        'flavor': 's3-compat', 'sts-enabled': True,
    },
    'storage-credential': {
        'type': 's3', 'credential-type': 'access-key',
        'access-key-id': 'seaweedfs-root-user',
        'secret-access-key': 'seaweedfs-root-password',
    },
}
r = requests.post(f'{MANAGEMENT_URL}/v1/warehouse', headers=auth(peter.token),
                  json=warehouse, timeout=30)
assert r.status_code in (200, 201, 409), (r.status_code, r.text)
wh = mlib.warehouse_id(peter.token)
print('warehouse id:', wh)

## 4 · Create the medallion namespaces

`raw` is a governed dataset of image *objects* in S3 (Bronze links to them by `s3://` URI);
Bronze & Silver are Iceberg; Gold is a Lance generic table — all under this one warehouse.

In [ ]:
for ns in (NS_RAW, NS_BRONZE, NS_SILVER, NS_GOLD):
    r = requests.post(f'{CATALOG_URL}/v1/{wh}/namespaces', headers=auth(peter.token),
                      json={'namespace': [ns]}, timeout=15)
    assert r.status_code in (200, 201, 409), (ns, r.status_code, r.text)
    print(f'namespace {ns}: {r.status_code}')

## 5 · Provision the service accounts

First touch provisions each as a catalog user and returns its id — we need the ids to
grant access. They have **no permissions yet**.

In [ ]:
pipeline_id   = mlib.agent_user_id(PIPELINE)
analyst_id    = mlib.agent_user_id(ANALYST)
contractor_id = mlib.agent_user_id(CONTRACTOR)
print('pipeline   ->', pipeline_id)
print('analyst    ->', analyst_id)
print('contractor ->', contractor_id)

## 6 · Grant access — the fine-grained governance

peter now hands out **differentiated** access. The governance boundary runs *through*
the medallion, per layer:

| principal | raw | bronze | silver | gold | why |
|---|---|---|---|---|---|
| `PIPELINE` | write | write | write | write | builds every layer |
| `ANALYST` (agent) | — | — | — | **read** | consumes the embeddings |
| `CONTRACTOR` (agent) | read | read | read | **denied** | may see source data, not the AI asset |

The contractor is a *legitimate collaborator* — it can read the source layers (raw images,
metadata, captions) — but Lakekeeper withholds the Gold embeddings. That single asymmetry
is the entire demo.

In [ ]:
# PIPELINE: warehouse-wide create + modify (cascades to every namespace/table).
mlib.grant_warehouse(peter.token, wh, pipeline_id, 'create')
mlib.grant_warehouse(peter.token, wh, pipeline_id, 'modify')

# Resolve namespace UUIDs (permission grants are keyed by id, not name).
raw_id    = mlib.namespace_id(peter.token, NS_RAW)
bronze_id = mlib.namespace_id(peter.token, NS_BRONZE)
silver_id = mlib.namespace_id(peter.token, NS_SILVER)
gold_id   = mlib.namespace_id(peter.token, NS_GOLD)

# ANALYST: read Gold only.
mlib.grant_namespace(peter.token, gold_id, analyst_id, 'select')

# CONTRACTOR: read the source layers (raw + bronze + silver), NOT Gold.
for nsid in (raw_id, bronze_id, silver_id):
    mlib.grant_namespace(peter.token, nsid, contractor_id, 'select')

print('grants applied. Contractor reads raw/bronze/silver but is denied Gold.')

---
Setup complete. Next: **01-build-medallion.ipynb** builds the layers, then
**02-agent-governed.ipynb** points the two agents at Gold.